# TensorBoard Viewer

Independent notebook with its own kernel, so the training notebook `autoTest.ipynb` is never blocked.

**No GPU needed.** Use Runtime -> Change runtime type -> CPU to save GPU quota.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Snapshot Drive event files to local SSD
Pointing TensorBoard directly at Drive runs into the FUSE read-side cache, so newly written events often do not show up. Copying the event files to `/content/local_tb` and pointing TB at the local copy avoids the problem entirely.

**Re-run this cell whenever you want the latest progress.** It pulls a fresh snapshot from Drive; the TB iframe will then auto-refresh from the local copy.

In [ ]:
import pathlib
import shutil

SRC = pathlib.Path(
    '/content/drive/MyDrive/autoTest/models/stage1_transformer/tensorboard'
)
DST = pathlib.Path('/content/local_tb')
DST.mkdir(parents=True, exist_ok=True)

# Mirror every run directory from Drive to local SSD.
# Overwrites existing local copies so re-running this cell pulls updates.
for run_dir in sorted(SRC.iterdir()):
    if not run_dir.is_dir():
        continue
    target = DST / run_dir.name
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(run_dir, target)
    print(f'copied: {run_dir.name}')

print()
!ls -la /content/local_tb/

## 3. Launch TensorBoard (reads from local SSD)
Once the iframe appears, in the bottom-left runs panel check the **latest timestamp** (e.g. `20260522_xxxxxx`) to see the current training session.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/local_tb --reload_interval 5